# client-streamable-http: Talking to the server over HTTP

**Over Streamable HTTP, the server runs on its own as a small web server, and the client sends each MCP message to one address, `/mcp`, as an HTTP POST.** Unlike stdio, you start the server yourself first, and many clients can use it at the same time.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 terminal: python server.py          this notebook (client)
 --------------------------          ----------------------
 http://127.0.0.1:8050/mcp  &lt;---  POST: one JSON-RPC message
                            ---&gt;  reply: JSON, or a stream
</pre>

Read `server.ipynb` and `client-stdio.ipynb` first. This notebook covers only what's different.

## 1. The basics: HTTP, an address, a port and an endpoint

- **HTTP:** The way browsers and web servers talk. The client sends a request, like GET to fetch something or POST to send data, and the server answers with a status code (200 means OK) and a body.
- **127.0.0.1:** The address that means "this computer".
- **Port 8050:** A numbered door on that computer. The server listens at door 8050.
- **Endpoint `/mcp`:** The path after the address and port. Every MCP message goes to this one path.

## 2. Starting the server

Open a terminal in this folder and start the server. With no word after `server.py`, it uses Streamable HTTP:

<div style="font-size: 0.85em">

```bash
conda activate MCP_env
python server.py
```

</div>

Leave it running while you run the cells below. Section 5 shows what it prints.

## 3. Connecting, listing the tools and calling `add`

`streamable_http_client(url)` is the HTTP version of `stdio_client`. It gives back three things instead of two:

- **`read` and `write`:** The same two streams as over stdio.
- **`get_session_id`:** A function that returns the session ID the server gives this client (section 4).

Everything from `ClientSession` down is the same as over stdio. No `errlog` is needed, because the server isn't a child of this notebook. It writes its logs in its own terminal.

In [1]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

url = "http://127.0.0.1:8050/mcp"

In [2]:
async with streamable_http_client(url) as (read, write, get_session_id):
    async with ClientSession(read, write) as session:
        init_result = await session.initialize()
        session_id = get_session_id()
        tools_result = await session.list_tools()
        result = await session.call_tool("add", {"a": 2, "b": 3})

print(result.content[0].text)

5


The result is the same `CallToolResult` as over stdio. Only the transport changed:

In [3]:
print(type(result))
print(result.model_dump_json(indent=2, exclude_none=True))

<class 'mcp.types.CallToolResult'>
{
  "content": [
    {
      "type": "text",
      "text": "5"
    }
  ],
  "structuredContent": {
    "result": 5
  },
  "isError": false
}


## 4. The session ID

In [4]:
print(session_id)

f766c8ccaaf9461f9a4967d5260e09be


In protocol 2025-11-25, which this SDK speaks, an HTTP server gives each client a session ID after the handshake. The client sends it back in an `Mcp-Session-Id` header on every later request, so the server knows which client is talking. stdio needs no ID, because each server process has exactly one client. The newest spec (2026-07-28) drops sessions. See the appendix of `../1-understanding-mcp/README.md`.

## 5. What the server saw

After its startup lines, the server's terminal printed this while the cells above ran. The session ID matches section 4:

<div style="font-size: 0.85em">

```text
Created new transport with session ID: f766c8ccaaf9461f9a4967d5260e09be
INFO:     127.0.0.1:49369 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49370 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49371 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49372 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type ListToolsRequest
INFO:     127.0.0.1:49373 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type CallToolRequest
Terminating session: f766c8ccaaf9461f9a4967d5260e09be
INFO:     127.0.0.1:49374 - "DELETE /mcp HTTP/1.1" 200 OK
```

</div>

Each `POST`, `GET` and `DELETE` line is one HTTP request:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 POST   200  initialize (the reply carries the session ID)
 POST   202  notifications/initialized (202: Received, no reply)
 GET    200  an extra stream for messages the server starts itself
 POST   200  tools/list
 POST   200  tools/call
 DELETE 200  the client ends the session
</pre>

A notification never gets a reply, which is why it gets a bare 202. The `127.0.0.1:` numbers at the start of each line are the client's side of each connection, and they change every time.

On Windows, the server's terminal may also print a `ConnectionResetError: [WinError 10054]` traceback near the end, when the client closes its connections. It's harmless, because the requests have all been answered by then, so it's left out of the log above.

## 6. stdio vs Streamable HTTP

- **Who starts the server:** Over stdio, the client does. Over HTTP, you do, and it keeps running after the client leaves.
- **How many clients:** A stdio server process has exactly one client. An HTTP server serves many.
- **Where it can run:** Over stdio, only on the same machine. Over HTTP, anywhere the client can reach: Docker (Part 5), another machine, or the cloud.
- **What the client needs:** A command for stdio, and a URL for HTTP.

## 7. The .py version

`client-streamable-http.py` does the same in a plain script, inside `async def main()` and `asyncio.run(main())`:

<div style="font-size: 0.85em">

```python
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

url = "http://127.0.0.1:8050/mcp"


async def main():
    async with streamable_http_client(url) as (read, write, get_session_id):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print(get_session_id())

            tools_result = await session.list_tools()
            tool = tools_result.tools[0]
            print(tool.name)
            print(tool.description)

            result = await session.call_tool("add", {"a": 2, "b": 3})
            print(result.content[0].text)


asyncio.run(main())
```

</div>

With the server running, run it from a second terminal with `python client-streamable-http.py`. The first line is that run's session ID:

<div style="font-size: 0.85em">

```text
004ee13136154577a353b2d57559bd47
add
Add two numbers together
5
```

</div>

## Say it out loud

> "Over Streamable HTTP, the server runs on its own, here at 127.0.0.1:8050, and every MCP message is an HTTP POST to one endpoint, `/mcp`. The server answers with JSON or a stream. In the protocol version I used, the server gives each client a session ID in the `Mcp-Session-Id` header, and the newest spec drops sessions. Only the connect line differs from stdio: `streamable_http_client` instead of `stdio_client`. I'd use it for a server shared by many clients, or one running in Docker or the cloud."